# 04 - Exploracion GEO para mapas de red CHEC

Este cuaderno analiza primero las capas geograficas en `data/GEO` antes de graficarlas. La idea es responder: que trae cada capa, que calidad espacial tiene, como se puede unir con `Indicadores_vano_v3.csv`, y que mapas conviene generar por circuito.

## 1. Setup

Las capas son shapefiles en WGS84 (`EPSG:4326`). Para calculos metricos se reproyectan temporalmente a un CRS estimado por `geopandas`.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import warnings

import geopandas as gpd
import pandas as pd
import folium

pd.set_option("display.max_columns", 120)
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

GEO_DIR = ROOT / "data" / "GEO"
EVENTS_PATH = ROOT / "data" / "Indicadores_vano_v3.csv"
OUTPUT_DIR = ROOT / "reports" / "geo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LINE_PATH = GEO_DIR / "MVLINSEC.shp"

print(f"GEO_DIR: {GEO_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

GEO_DIR: /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/data/GEO
OUTPUT_DIR: /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/reports/geo


## 2. Inventario de archivos

Antes de leer geometria conviene verificar que cada shapefile tenga sus archivos hermanos (`.shp`, `.shx`, `.dbf`, `.prj`, `.cpg`).

In [2]:
inventory = []
for path in sorted(GEO_DIR.glob("*")):
    inventory.append({
        "archivo": path.name,
        "extension": path.suffix.lower(),
        "tamano_mb": round(path.stat().st_size / 1024 / 1024, 3),
    })

df_inventory = pd.DataFrame(inventory)
display(df_inventory.sort_values(["archivo"]))

shp_bases = sorted(path.stem for path in GEO_DIR.glob("*.shp"))
required_ext = {".shp", ".shx", ".dbf", ".prj", ".cpg"}
for base in shp_bases:
    present = {path.suffix.lower() for path in GEO_DIR.glob(f"{base}.*")}
    missing = sorted(required_ext - present)
    print(f"{base}: faltantes requeridos={missing or 'ninguno'}")

,archivo,extension,tamano_mb
0,GDBCHEC_TRANSFOR.cpg,.cpg,0.000
1,GDBCHEC_TRANSFOR.dbf,.dbf,51.785
2,GDBCHEC_TRANSFOR.prj,.prj,0.000
3,GDBCHEC_TRANSFOR.sbn,.sbn,0.209
4,GDBCHEC_TRANSFOR.sbx,.sbx,0.013
5,GDBCHEC_TRANSFOR.shp,.shp,0.909
6,GDBCHEC_TRANSFOR.shp.xml,.xml,0.037
7,GDBCHEC_TRANSFOR.shx,.shx,0.165
8,MVLINSEC.cpg,.cpg,0.000
9,MVLINSEC.dbf,.dbf,90.833


GDBCHEC_TRANSFOR: faltantes requeridos=ninguno
MVLINSEC: faltantes requeridos=ninguno


## 3. Carga de capas y perfil basico

- `MVLINSEC`: vanos/tramos de red de media tension como `LineString`.

El analisis inicial revisa registros, CRS, tipo geometrico, limites espaciales, nulos y columnas candidatas para unir los vanos con la base analitica.

In [3]:
lineas = gpd.read_file(LINE_PATH)
lineas["FID_VANO_GEO"] = lineas["G3E_FID"]

layers = {
    "vanos_mv": lineas,
}

def profile_layer(name: str, gdf: gpd.GeoDataFrame) -> dict:
    non_geom = gdf.drop(columns=gdf.geometry.name)
    return {
        "layer": name,
        "rows": int(len(gdf)),
        "columns": int(len(gdf.columns)),
        "crs": str(gdf.crs),
        "geometry_types": gdf.geometry.geom_type.value_counts(dropna=False).to_dict(),
        "empty_geometries": int(gdf.geometry.is_empty.sum()),
        "null_geometries": int(gdf.geometry.isna().sum()),
        "bounds": [round(float(x), 6) for x in gdf.total_bounds],
        "top_null_columns": non_geom.isna().mean().sort_values(ascending=False).head(12).round(3).to_dict(),
    }

profiles = pd.DataFrame([profile_layer(name, gdf) for name, gdf in layers.items()])
display(profiles)

for name, gdf in layers.items():
    print(f"\n=== {name} ===")
    print(list(gdf.columns))
    display(gdf.drop(columns="geometry").head(5))

,layer,rows,columns,crs,geometry_types,empty_geometries,null_geometries,bounds,top_null_columns
0,vanos_mv,60053,45,EPSG:4326,{'LineString': 60053},0,0,"[-76.212507, 4.555535, -74.635654, 5.877015]","{'SERVICIO': 1.0, 'TIPO_AISLA': 0.808, 'TRABAJ..."



=== vanos_mv ===
['G3E_FID', 'CODIGO', 'CIRCUITO', 'TENSION', 'FASES', 'LONGITUD', 'LOCALIZACI', 'CODE_CONDU', 'CALIBRE_F', 'MATERIAL_F', 'TIPO_AISLA', 'AISLAMIENT', 'CODE_NEUTR', 'CALIBRE_NE', 'PROPIETARI', 'NODO1_ID', 'NODO2_ID', 'ESTADO', 'FECHA_COLO', 'FECHA_MODI', 'GLOBALID', 'NOMBRE_PRO', 'CLASIFICAC', 'FINALIDAD_', 'TIPO_PROYE', 'TRABAJO_PR', 'USUARIO_MO', 'EST_ESTABL', 'FECHA_INST', 'DEPARTAMEN', 'MUNICIPIO', 'TENSION_LO', 'ESTADO_EST', 'ELNODE1', 'ELNODE2', 'FECHA_REMP', 'CODIGO_OPE', 'FID_ANTERI', 'PROYECTO', 'SERVICIO', 'CIRCUITO_O', 'EST_OPERAT', 'SHAPE_LEN', 'geometry', 'FID_VANO_GEO']


,G3E_FID,CODIGO,CIRCUITO,TENSION,FASES,LONGITUD,LOCALIZACI,CODE_CONDU,CALIBRE_F,MATERIAL_F,TIPO_AISLA,AISLAMIENT,CODE_NEUTR,CALIBRE_NE,PROPIETARI,NODO1_ID,NODO2_ID,ESTADO,FECHA_COLO,FECHA_MODI,GLOBALID,NOMBRE_PRO,CLASIFICAC,FINALIDAD_,TIPO_PROYE,TRABAJO_PR,USUARIO_MO,EST_ESTABL,FECHA_INST,DEPARTAMEN,MUNICIPIO,TENSION_LO,ESTADO_EST,ELNODE1,ELNODE2,FECHA_REMP,CODIGO_OPE,FID_ANTERI,PROYECTO,SERVICIO,CIRCUITO_O,EST_OPERAT,SHAPE_LEN,FID_VANO_GEO
0,31362737,31362737,VBO23L15,13.2,AN,432.0,AEREO,5A-0624,2,ACSR,NaN,DESNUDO,5A-0624,2,CHEC,70773836,70773837,OPERACION,2020-08-14,2023-04-18,{EE9B2409-5556-4482-8AB3-14D06DFB7698},CHEC,RURAL,REEMPLAZO DE ACTIVOS SIN AUMENTO DE CAPACIDAD ...,T3,172168,EPM\LROLDANJ,CLOSED,2020-08-10,CALDAS,VITERBO,13.2_AEREO,OPERACION_CLOSED,NaN,NaN,NaT,L0033186,NaN,NaN,None,VBO23L15,CLOSED,0.003928,31362737
1,20537444,53069,ESM23L13,13.2,ABCN,179.3,AEREO,5A-0628,2/0,ACSR,NaN,DESNUDO,5A-0626,1/0,PARTICULAR,19044459,19044472,OPERACION,2019-07-20,2025-09-17,{BCF45E95-3255-498C-93C2-D3A931FB01D0},CHEC/GENERACION,RURAL,ANTIGUO,T2,NaN,LBEDOYQ,CLOSED,2005-10-31,CALDAS,CHINCHINA,13.2_AEREO,OPERACION_CLOSED,C29013,C29028,2005-10-31,L0045333,20537444,CT_LYM_FASE4,None,DESENERGIZADO,OPEN,0.001589,20537444
2,21153265,50704,VCT23L17,13.2,ABN,397.4,AEREO,5A-0624,2,ACSR,NaN,DESNUDO,5A-0624,2,PARTICULAR,19059967,19059968,OPERACION,2019-07-20,2025-09-17,{CB14F4BA-50CF-4624-815A-0782C3587C8B},PPP,RURAL,ANTIGUO,T2,NaN,VGARCIA,CLOSED,2004-06-03,CALDAS,VICTORIA,13.2_AEREO,OPERACION_CLOSED,L11842,L11843,2004-06-03,L0032244,21153265,CT_LYM_FASE3,None,VCT23L17,CLOSED,0.003524,21153265
3,20797000,13653,NSA23L14,13.2,ABN,204.6,AEREO,5A-0624,2,ACSR,NaN,DESNUDO,5A-0624,2,CHEC,19058053,19058054,OPERACION,2019-07-20,2025-09-17,{1CB2B7C8-164C-4A90-ACBE-B7863B647B05},NaN,RURAL,INSTALACIÓN DE ACTIVOS NUEVOS PARA ATENCIÓN DE...,T2,NaN,VGARCIA,CLOSED,1899-12-30,CALDAS,SAMANA,13.2_AEREO,OPERACION_CLOSED,E55270,E55271,1899-12-30,L0022013,20797000,NaN,None,NSA23L14,CLOSED,0.001848,20797000
4,20615857,4561,INS23L13,13.2,ABC,33.5,AEREO,5A-0622,4,ACSR,NaN,DESNUDO,NaN,NaN,PARTICULAR,19042975,19042976,OPERACION,2019-07-20,2025-09-17,{EE5D594D-7B1F-4A2C-AC33-9819250BEE1C},PPP,RURAL,INSTALACIÓN DE ACTIVOS NUEVOS PARA ATENCIÓN DE...,T2,NaN,SMARULAN,CLOSED,NaT,CALDAS,CHINCHINA,13.2_AEREO,OPERACION_CLOSED,C22753,C22754,NaT,L0018877,20615857,NaN,None,INS23L13,CLOSED,0.000650,20615857


## 4. Calidad espacial y medidas metricas

Las coordenadas estan en grados. Para medir longitudes se usa un CRS proyectado estimado. Esto permite comparar `LONGITUD`, `SHAPE_LEN` y longitud geometrica calculada.

In [4]:
metric_crs = lineas.estimate_utm_crs() or "EPSG:3116"
print(f"CRS metrico estimado: {metric_crs}")

lineas_metricas = lineas.to_crs(metric_crs).copy()
lineas["longitud_geom_m"] = lineas_metricas.geometry.length

quality_rows = []
for name, gdf in layers.items():
    quality_rows.append({
        "layer": name,
        "geometrias_validas": int(gdf.geometry.is_valid.sum()),
        "geometrias_invalidas": int((~gdf.geometry.is_valid).sum()),
        "geometrias_vacias": int(gdf.geometry.is_empty.sum()),
        "geometrias_nulas": int(gdf.geometry.isna().sum()),
    })

display(pd.DataFrame(quality_rows))

cols_len = [col for col in ["CIRCUITO", "FID_VANO_GEO", "CODIGO", "LONGITUD", "SHAPE_LEN", "longitud_geom_m"] if col in lineas.columns]
display(lineas[cols_len].describe(include="all"))

if "LONGITUD" in lineas.columns:
    comp = lineas[["CIRCUITO", "FID_VANO_GEO", "LONGITUD", "longitud_geom_m"]].copy()
    comp["LONGITUD"] = pd.to_numeric(comp["LONGITUD"], errors="coerce")
    comp["delta_m"] = comp["LONGITUD"] - comp["longitud_geom_m"]
    display(comp[["LONGITUD", "longitud_geom_m", "delta_m"]].describe())

CRS metrico estimado: EPSG:32618


,layer,geometrias_validas,geometrias_invalidas,geometrias_vacias,geometrias_nulas
0,vanos_mv,59776,277,0,0


,CIRCUITO,FID_VANO_GEO,CODIGO,LONGITUD,SHAPE_LEN,longitud_geom_m
count,60053,6.005300e+04,60053,60053.000000,60053.000000,60053.000000
unique,228,NaN,60053,NaN,NaN,NaN
top,DON23L13,NaN,31362737,NaN,NaN,NaN
freq,1377,NaN,1,NaN,NaN,NaN
mean,NaN,3.684274e+07,NaN,150.647778,0.001338,148.087567
std,NaN,6.802562e+07,NaN,181.971271,0.001678,185.681591
min,NaN,2.013043e+07,NaN,0.100000,0.000000,0.000000
25%,NaN,2.043656e+07,NaN,35.800000,0.000314,34.771758
50%,NaN,2.068783e+07,NaN,77.000000,0.000683,75.545405
75%,NaN,2.106039e+07,NaN,198.200000,0.001756,194.205221


,LONGITUD,longitud_geom_m,delta_m
count,60053.000000,60053.000000,60053.000000
mean,150.647778,148.087567,2.560211
std,181.971271,185.681591,51.728505
min,0.100000,0.000000,-11979.107625
25%,35.800000,34.771758,0.032829
50%,77.000000,75.545405,0.360849
75%,198.200000,194.205221,2.909075
max,2807.400000,11980.107625,1053.076229


## 5. Resumen operativo por circuito

Este resumen ayuda a decidir que circuitos graficar primero: cantidad de vanos/tramos, longitud de red y municipios cubiertos.

In [5]:
lineas_summary = (
    lineas.assign(longitud_geom_km=lineas["longitud_geom_m"] / 1000)
    .groupby("CIRCUITO", dropna=False)
    .agg(
        n_vanos_geo=("FID_VANO_GEO", "count"),
        longitud_geom_km=("longitud_geom_km", "sum"),
        municipios=("MUNICIPIO", lambda s: sorted({str(x) for x in s.dropna().unique()})),
    )
    .reset_index()
)

geo_circuit_summary = lineas_summary.copy()
geo_circuit_summary = geo_circuit_summary.sort_values("longitud_geom_km", ascending=False)
display(geo_circuit_summary.head(20))

summary_path = OUTPUT_DIR / "geo_resumen_circuitos.csv"
geo_circuit_summary.to_csv(summary_path, index=False)
print(f"Resumen guardado: {summary_path}")

,CIRCUITO,n_vanos_geo,longitud_geom_km,municipios
201,SLM23L13,828,246.421109,"[AGUADAS, MARULANDA, PENSILVANIA, SALAMINA]"
70,DON23L13,1377,213.822870,"[LA DORADA, VICTORIA]"
87,FIL23L13,820,168.788414,"[ARANZAZU, FILADELFIA, NEIRA, RIOSUCIO]"
120,MAZ23L14,513,167.220292,"[MANZANARES, MARULANDA, SALAMINA]"
172,PSV23L13,547,146.737967,"[NARIÑO, PENSILVANIA, SAMANA]"
155,NSA23L14,513,145.359577,"[NORCASIA, SAMANA]"
3,AGU23L15,634,144.095292,"[ABEJORRAL, AGUADAS, CARAMANTA, PACORA, SANTA ..."
99,HER23L16,789,138.417175,[SANTA ROSA DE CABAL]
157,PRA23L13,597,134.741831,"[AGUADAS, MARMATO, PACORA]"
227,VMA23L16,537,132.593827,"[SANTA ROSA DE CABAL, VILLAMARIA]"


Resumen guardado: /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/reports/geo/geo_resumen_circuitos.csv


## 6. Llaves candidatas para unir con `Indicadores_vano_v3.csv`

Hipotesis iniciales:

- Eventos `FID_VANO` -> vanos/tramos `MVLINSEC.G3E_FID`.
- Eventos `CIRCUITO` -> `MVLINSEC.CIRCUITO`.

La celda siguiente cuantifica cobertura antes de construir mapas con eventos.

In [6]:
def norm_id(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({"": pd.NA, "<NA>": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

event_cols = ["CIRCUITO", "FID_VANO", "UITI_VANO", "FECHA"]
events = pd.read_csv(EVENTS_PATH, usecols=lambda c: c in event_cols)
for col in ["FID_VANO", "CIRCUITO"]:
    if col in events.columns:
        events[col] = norm_id(events[col])

line_keys = lineas[["G3E_FID", "CIRCUITO"]].copy()
for col in line_keys.columns:
    line_keys[col] = norm_id(line_keys[col])

coverage = []
tests = [
    ("FID_VANO -> MVLINSEC.G3E_FID", events.get("FID_VANO"), line_keys["G3E_FID"]),
    ("CIRCUITO -> lineas.CIRCUITO", events.get("CIRCUITO"), line_keys["CIRCUITO"]),
]
for label, left, right in tests:
    if left is None:
        continue
    left_unique = set(left.dropna().astype(str).unique())
    right_unique = set(right.dropna().astype(str).unique())
    matched = left_unique & right_unique
    coverage.append({
        "union": label,
        "valores_eventos": len(left_unique),
        "valores_geo": len(right_unique),
        "valores_eventos_con_match": len(matched),
        "cobertura_eventos_unique": round(len(matched) / max(len(left_unique), 1), 4),
    })

display(pd.DataFrame(coverage).sort_values("cobertura_eventos_unique", ascending=False))

,union,valores_eventos,valores_geo,valores_eventos_con_match,cobertura_eventos_unique
1,CIRCUITO -> lineas.CIRCUITO,208,228,208,1.0000
0,FID_VANO -> MVLINSEC.G3E_FID,27390,60053,27305,0.9969


## 7. Agregacion de eventos sobre geometria

Si la cobertura de `FID_VANO -> lineas.G3E_FID` es buena, se puede pintar cada tramo por eventos, suma de `UITI_VANO` o promedio. Si no, se puede empezar por mapas a nivel circuito.

In [7]:
events_agg_vano = (
    events.assign(
        FID_VANO=norm_id(events["FID_VANO"]),
        UITI_VANO_NUM=pd.to_numeric(events.get("UITI_VANO"), errors="coerce"),
    )
    .groupby("FID_VANO", dropna=True)
    .agg(
        n_eventos=("FID_VANO", "size"),
        uiti_vano_total=("UITI_VANO_NUM", "sum"),
        uiti_vano_prom=("UITI_VANO_NUM", "mean"),
        circuitos_eventos=("CIRCUITO", lambda s: sorted({str(x) for x in s.dropna().unique()})),
    )
    .reset_index()
)

lineas_eventos = lineas.copy()
lineas_eventos["G3E_FID_NORM"] = norm_id(lineas_eventos["G3E_FID"])
lineas_eventos["FID_VANO_GEO"] = lineas_eventos["G3E_FID_NORM"]
lineas_eventos = lineas_eventos.merge(events_agg_vano, left_on="G3E_FID_NORM", right_on="FID_VANO", how="left")
lineas_eventos["n_eventos"] = lineas_eventos["n_eventos"].fillna(0).astype(int)
lineas_eventos["uiti_vano_total"] = lineas_eventos["uiti_vano_total"].fillna(0.0)

display(lineas_eventos[["CIRCUITO", "FID_VANO_GEO", "CODIGO", "n_eventos", "uiti_vano_total", "uiti_vano_prom"]].sort_values("uiti_vano_total", ascending=False).head(20))

,CIRCUITO,FID_VANO_GEO,CODIGO,n_eventos,uiti_vano_total,uiti_vano_prom
1290,BQE23L12,20306252,8804,36,40171.847128,1115.884642
22406,MTT23L12,20768787,43222,17,28601.161000,1682.421235
25633,BQE23L12,20306265,11043,36,27513.616115,764.267114
51855,DON23L12,20497969,55124,22,24457.321647,1111.696439
15651,BQE23L12,20306287,8849,18,24267.075000,1348.170833
42210,BQE23L12,20306261,8845,18,23365.258024,1298.069890
15650,BQE23L12,20306400,8829,18,21905.954189,1216.997455
4325,BQE23L12,20306401,55836,18,21397.657348,1188.758742
57416,BQE23L12,20981777,57253,18,21233.690625,1179.649479
17156,BQE23L12,20306479,67496,18,20856.567162,1158.698176


## 8. Mapas exploratorios

Primer mapa recomendado: circuito seleccionado con vanos/tramos coloreados por `uiti_vano_total`. Esto permite validar visualmente si la geometria corresponde a los vanos/eventos esperados antes de diseñar mapas definitivos.

In [8]:
CIRCUITO_MAPA = "DON23L13"

def style_line(feature):
    value = feature["properties"].get("uiti_vano_total") or 0
    if value > 0:
        color = "#dc2626"
        weight = 4
    else:
        color = "#2563eb"
        weight = 2
    return {"color": color, "weight": weight, "opacity": 0.75}

def make_circuit_map(circuito: str, max_features: int | None = None) -> folium.Map:
    g_lines = lineas_eventos[lineas_eventos["CIRCUITO"].astype(str).eq(str(circuito))].copy()
    if max_features is not None:
        g_lines = g_lines.head(max_features)
    if g_lines.empty:
        raise ValueError(f"No hay geometria para circuito {circuito}")

    bounds = g_lines.total_bounds
    center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]
    fmap = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")

    folium.GeoJson(
        g_lines[["FID_VANO_GEO", "CODIGO", "CIRCUITO", "n_eventos", "uiti_vano_total", "geometry"]],
        name="Vanos / tramos MV",
        style_function=style_line,
        tooltip=folium.GeoJsonTooltip(fields=["FID_VANO_GEO", "CODIGO", "CIRCUITO", "n_eventos", "uiti_vano_total"]),
    ).add_to(fmap)
    folium.LayerControl().add_to(fmap)
    fmap.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    return fmap

mapa = make_circuit_map(CIRCUITO_MAPA)
map_path = OUTPUT_DIR / f"mapa_geo_{CIRCUITO_MAPA}.html"
mapa.save(map_path)
print(f"Mapa guardado: {map_path}")
mapa

Mapa guardado: /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/reports/geo/mapa_geo_DON23L13.html


## 9. Como decidir que graficar despues

Use esta lista antes de construir mapas finales:

1. **Cobertura de llaves:** confirmar si `FID_VANO -> G3E_FID` cubre la mayoria de vanos con eventos. Si no, usar `CIRCUITO` como primer nivel.
2. **Calidad geometrica:** revisar geometrias vacias/invalidas y longitudes atipicas.
3. **Escala del mapa:** para red completa usar agregacion por circuito; para diagnostico usar circuito individual.
4. **Variable visual:** empezar con `n_eventos`, `uiti_vano_total` y `uiti_vano_prom` sobre vanos/tramos.
5. **Comparacion con cuadernos existentes:** los mapas deben complementar, no reemplazar, los graficos de puntos criticos e inferencia.

Mapas candidatos:

- Red por circuito con color por `UITI_VANO` total.
- Circuito seleccionado con vanos/tramos con eventos resaltados.
- Mapa de calidad de datos: tramos sin evento asociado, sin circuito o con longitud inconsistente.